# DFT Validation — Notebook A (Al + Ti + references)

**Runs 15 QE SCF calculations:** undoped (2) + Li metal (1) + Al (6) + Ti (6)

Run Notebook B in a separate Colab tab simultaneously for Mg + Ga + Fe.

**Estimated runtime:** ~1.5–2.5 hours on Colab CPU

In [ ]:
# Cell 1: Install Quantum ESPRESSO
!apt-get -qq update
!apt-get -qq install -y quantum-espresso
!which pw.x && echo 'QE installed successfully'

In [ ]:
# Cell 2: Mount Google Drive and set up directories
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil

# Source directory on Drive (upload dft_sqs_validation/ folder here)
DRIVE_DIR = '/content/drive/MyDrive/dft_sqs_validation'
WORK_DIR = '/content/dft_work_A'

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f'{WORK_DIR}/pseudo', exist_ok=True)
os.makedirs(f'{WORK_DIR}/outputs', exist_ok=True)

# Copy pseudopotentials
for f in os.listdir(f'{DRIVE_DIR}/pseudo'):
    if f.endswith('.UPF'):
        shutil.copy2(f'{DRIVE_DIR}/pseudo/{f}', f'{WORK_DIR}/pseudo/{f}')
        print(f'  Copied {f}')

# Copy QE input files for this notebook's batch
BATCH_A = [
    'undoped_lith', 'undoped_delith', 'li_metal',
    'Al_ord_lith', 'Al_ord_delith',
    'Al_sqs0_lith', 'Al_sqs0_delith',
    'Al_sqs1_lith', 'Al_sqs1_delith',
    'Ti_ord_lith', 'Ti_ord_delith',
    'Ti_sqs0_lith', 'Ti_sqs0_delith',
    'Ti_sqs1_lith', 'Ti_sqs1_delith',
]

for name in BATCH_A:
    src = f'{DRIVE_DIR}/qe_sqs_inputs/{name}.in'
    dst = f'{WORK_DIR}/{name}.in'
    shutil.copy2(src, dst)
    print(f'  Copied {name}.in')

# Copy manifest
shutil.copy2(f'{DRIVE_DIR}/qe_sqs_inputs/manifest.json', f'{WORK_DIR}/manifest.json')

print(f'\nReady: {len(BATCH_A)} calculations to run')

In [ ]:
# Cell 3: Fix pseudo_dir path AND speed up SCF convergence
import glob

for inp_file in glob.glob(f'{WORK_DIR}/*.in'):
    with open(inp_file, 'r') as f:
        content = f.read()
    
    # Fix pseudo_dir path
    content = content.replace("pseudo_dir = './pseudo'", f"pseudo_dir = '{WORK_DIR}/pseudo'")
    
    # Speed up SCF: mixing_beta 0.1 -> 0.35 (original too conservative for Colab)
    content = content.replace('mixing_beta = 0.1', 'mixing_beta = 0.35')
    
    # Reduce max steps (converges faster with higher mixing_beta)
    content = content.replace('electron_maxstep = 500', 'electron_maxstep = 200')
    
    # Add better initial guess if not present
    if 'startingwfc' not in content:
        content = content.replace(
            "  conv_thr = 1e-05",
            "  conv_thr = 1e-05\n  startingwfc = 'atomic+random'\n  startingpot = 'atomic'"
        )
    
    with open(inp_file, 'w') as f:
        f.write(content)

print('Patched all input files:')
print('  pseudo_dir  -> local work dir')
print('  mixing_beta -> 0.35 (was 0.1)')
print('  electron_maxstep -> 200 (was 500)')
print('  Added startingwfc/startingpot')

# Clear old failed checkpoints so Cell 4 retries everything
import os
ckpt = f'{DRIVE_DIR}/results/checkpoint_A.json'
if os.path.exists(ckpt):
    os.remove(ckpt)
    print(f'\nCleared old checkpoint: {ckpt}')
else:
    print('\nNo old checkpoint to clear')

In [ ]:
# Cell 4: Run all QE calculations WITH per-calc checkpointing to Drive
#
# RESUME-SAFE: If Colab disconnects, re-run this cell.
# It loads the checkpoint from Drive and skips completed calcs.
#
# NOTE: Uses stdin redirect (not -i flag) for QE v6.7 compatibility on Colab.

import subprocess
import time
import json
import shutil

CHECKPOINT_DIR = f'{DRIVE_DIR}/results'
CHECKPOINT_FILE = f'{CHECKPOINT_DIR}/checkpoint_A.json'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Load existing checkpoint (resume after disconnect) ──
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        results = json.load(f)
    completed = [k for k, v in results.items() if v.get('converged') is not None]
    print(f'RESUMING: Found checkpoint with {len(completed)} completed calcs')
    print(f'  Already done: {completed}')
else:
    results = {}
    print('Starting fresh (no checkpoint found)')

failed = [k for k, v in results.items() if v.get('converged') == False]
t_total_start = time.time()

for i, name in enumerate(BATCH_A):
    # ── Skip already-completed calcs ──
    if name in results and results[name].get('converged') is not None:
        status = '✓' if results[name]['converged'] else '✗'
        print(f'[{i+1}/{len(BATCH_A)}] {name} — {status} SKIPPED (already in checkpoint)')
        continue

    print(f'\n[{i+1}/{len(BATCH_A)}] Running {name}...', flush=True)
    t_start = time.time()
    
    inp_file = f'{WORK_DIR}/{name}.in'
    out_file = f'{WORK_DIR}/outputs/{name}.out'
    
    try:
        # Use stdin redirect — QE v6.7 on Colab doesn't support -i flag
        with open(inp_file) as fin:
            result = subprocess.run(
                ['pw.x'],
                stdin=fin,
                capture_output=True, text=True,
                timeout=7200,  # 2 hour max per calc (safety net)
                cwd=WORK_DIR
            )
        
        # Save QE output file locally
        with open(out_file, 'w') as f:
            f.write(result.stdout)
            if result.stderr:
                f.write('\n--- STDERR ---\n')
                f.write(result.stderr)
        
        if 'convergence has been achieved' in result.stdout:
            for line in result.stdout.split('\n'):
                if '!' in line and 'total energy' in line:
                    energy_ry = float(line.split('=')[1].split('Ry')[0].strip())
                    energy_ev = energy_ry * 13.605698
                    results[name] = {
                        'energy_Ry': energy_ry,
                        'energy_eV': energy_ev,
                        'converged': True,
                        'time_s': time.time() - t_start
                    }
                    print(f'  ✓ Converged: E = {energy_ry:.6f} Ry = {energy_ev:.4f} eV ({time.time()-t_start:.0f}s)')
                    break
        else:
            elapsed = time.time() - t_start
            print(f'  ✗ DID NOT CONVERGE ({elapsed:.0f}s)')
            # Show last few lines for debugging
            last_lines = result.stdout.strip().split('\n')[-5:]
            for l in last_lines:
                print(f'    {l}')
            failed.append(name)
            results[name] = {'converged': False, 'time_s': elapsed}
            
    except subprocess.TimeoutExpired:
        print(f'  ✗ TIMEOUT (>2 hours)')
        failed.append(name)
        results[name] = {'converged': False, 'time_s': 7200, 'error': 'timeout'}
    except Exception as e:
        print(f'  ✗ ERROR: {e}')
        failed.append(name)
        results[name] = {'converged': False, 'error': str(e)}
    
    # ── CHECKPOINT: Save to Drive after EVERY calc ──
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(results, f, indent=2)
    # Also save the QE output file to Drive immediately
    if os.path.exists(out_file):
        shutil.copy2(out_file, f'{CHECKPOINT_DIR}/{name}.out')
    
    n_done = sum(1 for v in results.values() if v.get('converged') is not None)
    n_ok = sum(1 for v in results.values() if v.get('converged') == True)
    print(f'  💾 Checkpoint saved to Drive: {n_ok} converged, {n_done}/{len(BATCH_A)} total')

t_total = time.time() - t_total_start
n_converged = sum(1 for v in results.values() if v.get('converged') == True)
print(f'\n{"="*60}')
print(f'DONE: {n_converged}/{len(BATCH_A)} converged in {t_total/60:.1f} min')
if failed:
    print(f'FAILED: {failed}')
print(f'\nAll results checkpointed to: {CHECKPOINT_FILE}')
print(f'If Colab disconnected, just re-run this cell to resume.')

In [ ]:
# Cell 5: Save results to Google Drive
import shutil

# Save results JSON
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)
shutil.copy2(f'{WORK_DIR}/results_A.json', f'{DRIVE_DIR}/results/results_A.json')

# Save all output files
for out_file in glob.glob(f'{WORK_DIR}/outputs/*.out'):
    shutil.copy2(out_file, f'{DRIVE_DIR}/results/{os.path.basename(out_file)}')

print(f'Saved to {DRIVE_DIR}/results/')
print('Run Notebook B for Mg + Ga + Fe results.')
print('Then combine with the analysis notebook.')

In [ ]:
# Cell 6: Quick sanity check — compute undoped voltage
if 'undoped_lith' in results and 'undoped_delith' in results and 'li_metal' in results:
    if results['undoped_lith']['converged'] and results['undoped_delith']['converged'] and results['li_metal']['converged']:
        E_lith = results['undoped_lith']['energy_eV']
        E_delith = results['undoped_delith']['energy_eV']
        E_li = results['li_metal']['energy_eV'] / 2  # per atom
        n_li = 18
        V = -(E_delith - E_lith + n_li * E_li) / n_li
        print(f'Undoped LiCoO2 DFT voltage: {V:.3f} V')
        print(f'Expected (experiment): ~3.9 V')
        print(f'Deviation: {abs(V - 3.9):.3f} V')
    else:
        print('Some reference calcs did not converge — check outputs')
else:
    print('Reference calculations not found in results')